# Citation 

Claude API에는 문서에 관한 질문에 답할 때 Claude가 상세한 citation을 제공할 수 있도록 citation 지원 기능이 있습니다. citation은 많은 LLM 기반 application에서 사용자가 응답 정보의 출처를 추적하고 확인하는 데 도움이 되는 유용한 기능입니다.

Citation을 지원하는 model:
* `claude-sonnet-4-6`
* `claude-3-5-haiku-20241022`

citation 기능은 프롬프트 기반 citation 기법의 대안입니다. 이 기능을 사용하면 다음과 같은 이점이 있습니다.
- 프롬프트 기반 기법은 Claude가 인용하려는 source document의 전체 인용문을 출력하도록 요구하는 경우가 많습니다. 이는 output token과 비용을 증가시킵니다.
- citation 기능은 유효한 source로 제공하지 않은 문서나 위치를 가리키는 citation을 반환하지 않습니다.
- 테스트에서 citation 기능은 프롬프트 기반 기법보다 더 높은 recall과 precision으로 citation을 생성했습니다.

citation 문서는 [여기](https://docs.claude.com/en/docs/build-with-claude/citations)에서 볼 수 있습니다.

## 설정

먼저 필요한 library를 설치하고 Anthropic client를 초기화합니다. 

In [ ]:
!pip install anthropic  --quiet

In [56]:
import json
import os

import anthropic

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
# ANTHROPIC_API_KEY = "" # 여기에 API key를 넣으세요!

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 문서 유형

Citation은 서로 다른 세 문서 유형을 지원합니다. 출력되는 citation 유형은 인용하는 문서 유형에 따라 달라집니다.

* Plain text document citation → 문자 위치 형식
* PDF document citation → 페이지 위치 형식
* Custom content document citation → content block 위치 형식

아래 예제에서 각각을 사용하는 방법을 살펴봅니다.

### Plain Text 문서

plain text document citation에서는 문서를 raw text로 model에 제공합니다. 문서를 하나 또는 여러 개 제공할 수 있습니다. 이 텍스트는 자동으로 문장 단위로 chunking됩니다. model은 필요에 따라 이 문장들을 인용합니다. 한 citation에서 여러 문장을 한꺼번에 인용할 수 있지만 문장보다 작은 텍스트는 인용하지 않습니다.

API 응답에는 출력 텍스트와 함께 모든 citation의 구조화된 data가 포함됩니다. 

가상의 회사 PetWorld의 help center 고객 chatbot을 사용하는 전체 예제를 보겠습니다.

In [57]:
# 모든 help center article을 읽고 문서 목록 생성
articles_dir = "./data/help_center_articles"
documents = []

for filename in sorted(os.listdir(articles_dir)):
    if filename.endswith(".txt"):
        with open(os.path.join(articles_dir, filename)) as f:
            content = f.read()
            # 제목과 본문으로 분리
            title_line, body = content.split("\n", 1)
            title = title_line.replace("title: ", "")
            documents.append(
                {
                    "type": "document",
                    "source": {"type": "text", "media_type": "text/plain", "data": body},
                    "title": title,
                    "citations": {"enabled": True},
                }
            )

QUESTION = "I just checked out, where is my order tracking number? Track package is not available on the website yet for my order."

# content에 질문 추가
content = documents

response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    system="You are a customer support bot working for PetWorld. Your task is to provide short, helpful answers to user questions. Since you are in a chat interface avoid providing extra details. You will be given access to PetWorld's help center articles to help you answer questions.",
    messages=[
        {"role": "user", "content": documents},
        {
            "role": "user",
            "content": [{"type": "text", "text": f"Here is the user's question: {QUESTION}"}],
        },
    ],
)


def visualize_raw_response(response):
    raw_response = {"content": []}

    print("\n" + "=" * 80 + "\nRaw response:\n" + "=" * 80)

    for content in response.content:
        if content.type == "text":
            block = {"type": "text", "text": content.text}
            if hasattr(content, "citations") and content.citations:
                block["citations"] = []
                for citation in content.citations:
                    citation_dict = {
                        "type": citation.type,
                        "cited_text": citation.cited_text,
                        "document_title": citation.document_title,
                    }
                    if citation.type == "page_location":
                        citation_dict.update(
                            {
                                "start_page_number": citation.start_page_number,
                                "end_page_number": citation.end_page_number,
                            }
                        )
                    block["citations"].append(citation_dict)
            raw_response["content"].append(block)

    return json.dumps(raw_response, indent=2)


print(visualize_raw_response(response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Based on the documentation, I can explain why you don't see tracking yet: "
    },
    {
      "type": "text",
      "text": "You'll receive an email with your tracking number once your order ships. If you don't receive a tracking number within 48 hours of your order confirmation, please contact our customer support team for assistance.",
      "citations": [
        {
          "type": "char_location",
          "cited_text": "Once your order ships, you'll receive an email with a tracking number. ",
          "document_title": "Order Tracking Information"
        },
        {
          "type": "char_location",
          "cited_text": "If you haven't received a tracking number within 48 hours of your order confirmation, please contact our customer support team.",
          "document_title": "Order Tracking Information"
        }
      ]
    },
    {
      "type": "text",
      "text": "\n\nSince you just checked

#### Citation 시각화
citation data를 활용하면 다음과 같은 UI를 만들 수 있습니다.

1. 정보가 어디서 왔는지 사용자에게 정확히 표시
2. source document에 직접 link
3. 맥락 속에서 인용된 텍스트 강조
4. 투명한 출처 제공을 통한 신뢰 형성

아래는 Claude의 구조화된 citation을 학술 논문과 비슷한 번호가 붙은 reference가 포함된 읽기 쉬운 형식으로 바꾸는 간단한 시각화 함수입니다.

함수는 Claude의 response 객체를 받아 다음을 출력합니다.
- 번호가 붙은 citation marker가 있는 텍스트(예: "답변 [1]에는 이 사실 [2]이 포함됩니다")
- 각 인용 텍스트와 source document를 보여 주는 번호가 붙은 reference 목록

In [58]:
def visualize_citations(response):
    """
    response 객체를 받아 번호가 붙은 citation이 있는 문자열을 반환합니다.
    출력 예제: "일반 텍스트 답변입니다 [1][2] 추가 텍스트입니다 [3]"
    아래에 citation 목록이 있습니다.
    """
    # 고유 citation을 저장할 dictionary
    citations_dict = {}
    citation_counter = 1

    # 최종 서식 지정 텍스트
    formatted_text = ""
    citations_list = []

    print("\n" + "=" * 80 + "\nFormatted response:\n" + "=" * 80)

    for content in response.content:
        if content.type == "text":
            text = content.text
            if hasattr(content, "citations") and content.citations:
                # 텍스트에 나타나는 순서로 citation 정렬
                def get_sort_key(citation):
                    if hasattr(citation, "start_char_index"):
                        return citation.start_char_index
                    elif hasattr(citation, "start_page_number"):
                        return citation.start_page_number
                    elif hasattr(citation, "start_block_index"):
                        return citation.start_block_index
                    return 0  # fallback

                sorted_citations = sorted(content.citations, key=get_sort_key)

                # 각 citation 처리
                for citation in sorted_citations:
                    doc_title = citation.document_title
                    cited_text = citation.cited_text.replace("\n", " ").replace("\r", " ")
                    # 생겼을 수 있는 중복 공백 제거
                    cited_text = " ".join(cited_text.split())

                    # 이 citation의 고유 key 생성
                    citation_key = f"{doc_title}:{cited_text}"

                    # 새 citation이면 dictionary에 추가
                    if citation_key not in citations_dict:
                        citations_dict[citation_key] = citation_counter
                        citations_list.append(
                            f'[{citation_counter}] "{cited_text}" found in "{doc_title}"'
                        )
                        citation_counter += 1

                    # 텍스트에 citation 번호 추가
                    citation_num = citations_dict[citation_key]
                    text += f" [{citation_num}]"

            formatted_text += text

    # 서식 지정 텍스트와 citation 목록 결합
    final_output = formatted_text + "\n\n" + "\n".join(citations_list)
    return final_output


formatted_response = visualize_citations(response)
print(formatted_response)


Formatted response:
Based on the documentation, I can explain why you don't see tracking yet: You'll receive an email with your tracking number once your order ships. If you don't receive a tracking number within 48 hours of your order confirmation, please contact our customer support team for assistance. [1] [2]

Since you just checked out, your order likely hasn't shipped yet. Once it ships, you'll receive the tracking information via email.

[1] "Once your order ships, you'll receive an email with a tracking number." found in "Order Tracking Information"
[2] "If you haven't received a tracking number within 48 hours of your order confirmation, please contact our customer support team." found in "Order Tracking Information"


### PDF 문서

PDF를 사용할 때 Claude는 특정 페이지 번호를 참조하는 citation을 제공하여 정보 출처를 쉽게 추적할 수 있게 합니다. PDF citation은 다음과 같이 작동합니다.

- PDF 문서 내용을 base64로 인코딩한 data로 제공
- 텍스트는 자동으로 문장 단위로 chunking됨
- citation에는 정보가 발견된 페이지 번호(1부터 시작)가 포함됨
- model은 한 citation에서 여러 문장을 함께 인용할 수 있지만 문장보다 작은 텍스트는 인용하지 않음
- 이미지는 처리되지만 현재는 텍스트 내용만 인용 가능

아래는 Constitutional AI 논문으로 PDF citation을 보여 주는 예제입니다.

In [59]:
import base64
import json

# PDF 읽기 및 인코딩
pdf_path = "data/Constitutional AI.pdf"
with open(pdf_path, "rb") as f:
    pdf_data = base64.b64encode(f.read()).decode()

pdf_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_data},
                    "title": "Constitutional AI Paper",
                    "citations": {"enabled": True},
                },
                {"type": "text", "text": "What is the main idea of Constitutional AI?"},
            ],
        }
    ],
)

print(visualize_raw_response(pdf_response))
print(visualize_citations(pdf_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Based on the paper, here are the key aspects of Constitutional AI:\n\n"
    },
    {
      "type": "text",
      "text": "Constitutional AI is a method for training a harmless AI assistant through self-improvement, without any human labels identifying harmful outputs. The only human oversight is provided through a list of rules or principles, hence the name \"Constitutional AI\".",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "We experiment with methods for training a harmless AI assistant through self\u0002improvement, without any human labels identifying harmful outputs. The only human\r\noversight is provided through a list of rules or principles, and so we refer to the method as\r\n\u2018Constitutional AI\u2019. ",
          "document_title": "Constitutional AI Paper",
          "start_page_number": 1,
          "end_page_number": 2
        }
      ]
    },
    {


### Custom Content 문서

plain text 문서는 자동으로 문장 단위로 chunking되지만, custom content 문서는 citation의 세분화 정도를 완전히 제어할 수 있게 합니다. 이 API 형식으로 다음을 할 수 있습니다.

* 어떤 크기든 자신의 chunk 정의
* 최소 citation 단위 제어
* 문장 chunking이 잘 맞지 않는 문서에 최적화

아래 예제는 위 plain text 예제와 같은 help center article을 사용하지만 문장 수준 citation을 허용하는 대신 각 article을 단일 chunk로 취급합니다. 이는 문서 유형의 선택이 citation 동작과 세분화 정도에 미치는 영향을 보여 줍니다. `cited_text`가 source article의 한 문장이 아니라 전체 article임을 볼 수 있습니다.

In [60]:
# 모든 help center article을 읽고 custom content 문서 목록 생성
articles_dir = "./data/help_center_articles"
documents = []

for filename in sorted(os.listdir(articles_dir)):
    if filename.endswith(".txt"):
        with open(os.path.join(articles_dir, filename)) as f:
            content = f.read()
            # 제목과 본문으로 분리
            title_line, body = content.split("\n", 1)
            title = title_line.replace("title: ", "")

            documents.append(
                {
                    "type": "document",
                    "source": {"type": "content", "content": [{"type": "text", "text": body}]},
                    "title": title,
                    "citations": {"enabled": True},
                }
            )

QUESTION = "I just checked out, where is my order tracking number? Track package is not available on the website yet for my order."

custom_content_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    system="You are a customer support bot working for PetWorld. Your task is to provide short, helpful answers to user questions. Since you are in a chat interface avoid providing extra details. You will be given access to PetWorld's help center articles to help you answer questions.",
    messages=[
        {"role": "user", "content": documents},
        {
            "role": "user",
            "content": [{"type": "text", "text": f"Here is the user's question: {QUESTION}"}],
        },
    ],
)

print(visualize_raw_response(custom_content_response))
print(visualize_citations(custom_content_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "You should receive an email with your tracking number once your order ships. If it's been less than 48 hours since your order confirmation, please wait as the tracking number may not be available yet. If you haven't received a tracking number after 48 hours, please contact our customer support team for assistance.",
      "citations": [
        {
          "type": "content_block_location",
          "cited_text": "Once your order ships, you'll receive an email with a tracking number. To track your package, log in to your PetWorld account and go to \"Order History.\" Click on the order you want to track and select \"Track Package.\" This will show you the current status and estimated delivery date. You can also enter the tracking number directly on our shipping partner's website for more detailed information. If you haven't received a tracking number within 48 hours of your order confirmation, please contact our 

### Context Field 사용

`context` field는 Claude가 응답을 생성할 때 사용할 수 있지만 인용되지는 않는 문서의 추가 정보를 제공할 수 있게 합니다. 다음에 유용합니다.

* 문서 metadata 제공(예: 발행일, 저자)
* [Contextual retrieval](https://www.anthropic.com/news/contextual-retrieval)
* 직접 인용해서는 안 되는 사용 지침이나 맥락 포함

아래 예제에서는 context field에 경고가 있는 loyalty program article을 제공합니다. Claude가 context의 정보로 응답을 구성할 수 있지만 context field 내용은 인용할 수 없다는 점을 살펴보세요.

In [61]:
import json

# context field가 있는 문서 생성
document = {
    "type": "document",
    "source": {
        "type": "text",
        "media_type": "text/plain",
        "data": "PetWorld offers a loyalty program where customers earn 1 point for every dollar spent. Once you accumulate 100 points, you'll receive a $5 reward that can be used on your next purchase. Points expire 12 months after they are earned. You can check your point balance in your account dashboard or by asking customer service.",
    },
    "title": "Loyalty Program Details",
    "context": "WARNING: This article has not been updated in 12 months. Content may be out of date. Be sure to inform the user this content may be incorrect after providing guidance.",
    "citations": {"enabled": True},
}

QUESTION = "How does PetWorld's loyalty program work? When do points expire?"

context_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    messages=[{"role": "user", "content": [document, {"type": "text", "text": QUESTION}]}],
)

print(visualize_raw_response(context_response))
print(visualize_citations(context_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Let me explain PetWorld's loyalty program based on the provided information:\n\n"
    },
    {
      "type": "text",
      "text": "PetWorld's loyalty program is straightforward - you earn 1 point for every dollar you spend. These points can be redeemed once you reach 100 points, which will get you a $5 reward that you can use on your next purchase.",
      "citations": [
        {
          "type": "char_location",
          "cited_text": "PetWorld offers a loyalty program where customers earn 1 point for every dollar spent. Once you accumulate 100 points, you'll receive a $5 reward that can be used on your next purchase. ",
          "document_title": "Loyalty Program Details"
        }
      ]
    },
    {
      "type": "text",
      "text": "\n\n"
    },
    {
      "type": "text",
      "text": "Points have an expiration period of 12 months from the date they are earned.",
      "citations": [
        {
   

### PDF 강조 표시

PDF citation의 제약 중 하나는 페이지 번호만 반환한다는 점입니다. third-party library로 반환된 인용 텍스트와 페이지 내용을 매칭하여 인용된 내용에 주의를 끌 수 있습니다. 이 셀은 Claude와 PyMuPDF를 사용한 PDF citation 강조 표시로 새로운 주석이 추가된 PDF를 만드는 모습을 보여 줍니다.

In [62]:
import fitz  # PyMuPDF

# 경로 설정 및 PDF 읽기
pdf_path = "data/Amazon-com-Inc-2023-Shareholder-Letter.pdf"
output_pdf_path = "data/Amazon-com-Inc-2023-Shareholder-Letter-highlighted.pdf"

# PDF 읽기 및 인코딩
with open(pdf_path, "rb") as f:
    pdf_data = base64.b64encode(f.read()).decode()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_data},
                    "title": "Amazon 2023 Shareholder Letter",
                    "citations": {"enabled": True},
                },
                {
                    "type": "text",
                    "text": "What was Amazon's total revenue in 2023 and how much did it grow year-over-year?",
                },
            ],
        }
    ],
)

print(visualize_raw_response(response))

# PDF citation 수집
pdf_citations = []
for content in response.content:
    if hasattr(content, "citations") and content.citations:
        for citation in content.citations:
            if citation.type == "page_location":
                pdf_citations.append(citation)

doc = fitz.open(pdf_path)

# 각 citation 처리
for citation in pdf_citations:
    if citation.type == "page_location":
        text_to_find = citation.cited_text.replace("\u0002", "")
        start_page = citation.start_page_number - 1  # 0부터 시작하는 index로 변환
        end_page = citation.end_page_number - 2

        # citation 범위의 각 페이지 처리
        for page_num in range(start_page, end_page + 1):
            page = doc[page_num]

            text_instances = page.search_for(text_to_find.strip())

            if text_instances:
                print(f"Found cited text on page {page_num + 1}")
                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.set_colors({"stroke": (1, 1, 0)})  # 노란색 강조 표시
                    highlight.update()
            else:
                print(f"{text_to_find} not found on page {page_num + 1}")

# 새 PDF 저장
doc.save(output_pdf_path)
doc.close()

print(f"\nCreated highlighted PDF at: {output_pdf_path}")


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "According to the letter, "
    },
    {
      "type": "text",
      "text": "Amazon's total revenue grew 12% year-over-year (\"YoY\") from $514B to $575B in 2023",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "In 2023, Amazon\u2019s total revenue grew 12% year-over-year (\u201cYoY\u201d) from $514B to $575B. ",
          "document_title": "Amazon 2023 Shareholder Letter",
          "start_page_number": 1,
          "end_page_number": 2
        }
      ]
    },
    {
      "type": "text",
      "text": ".\n\nBreaking this down by segment:\n"
    },
    {
      "type": "text",
      "text": "\n- North America revenue increased 12% YoY from $316B to $353B\n- International revenue grew 11% YoY from $118B to $131B  \n- AWS revenue increased 13% YoY from $80B to $91B",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "By segment, Nor